In [ ]:
## Import module dependencies
import pandas as pd #for manipulating data tables and csv files
import numpy as np #for mathematical manipulations
import matplotlib.pyplot as plt #for graphical plotting

## Importing various items from scikitlearn - a more intuitive, introductory toolbox for Machine Learning in Python

# For acquiring and preprocessing data
from sklearn.datasets import fetch_openml 
from sklearn.model_selection import train_test_split

# Various ML architectures/models we can test for our problem
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Before we start, let's define some functions which will make our life a little easier

## First, let's define a function to remove invalid/empty entries from our data 

In [ ]:
def removeNaNs(X, y):
    tmp = pd.notna(X)
    for row in range(len(X)):
        for x in tmp.iloc[row]:
            if x is False:
                X = X.drop(row)
                y = y.drop(row)
                break
            
    return X, y

## Now, let's define some functions which we can call to test and assess how good our machine learning models are for the problem

In [ ]:
def plot_roc_curve(fpr, tpr):
    auc = np.trapz(tpr, x=fpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f'model AUC = {auc:.2f}')
    plt.plot([0,1],[0,1], "r--", label="random classifier")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.legend()    
    plt.show()
    
def plot_confusion_matrix(prediction, Y_true):
    plt.figure()
    hist, xedge, yedge, im = plt.hist2d(Y_true, prediction, bins=(2,2), range=[[0,1],[0,1]])
    plt.xlabel("Truth label")
    plt.ylabel("Predicted label")
    plt.xticks(ticks=[0.25,0.75], labels=["0 (died)", "1 (survived)"])
    plt.yticks(ticks=[0.25,0.75], labels=["0 (died)", "1 (survived)"])
    for i in range(len(xedge)-1):
        for j in range(len(yedge)-1):
            plt.text(xedge[i]+0.25, yedge[j]+0.25, hist[i,j], 
                     backgroundcolor="white", color="black", weight="bold",
                     ha="center", va="center")
    plt.colorbar()
    plt.show()
    
def roc_curve(model, X_test, Y_test):
    tot_pos = sum(Y_test)
    tot_neg = len(Y_test) - tot_pos
    prediction = model.predict(X_test)
    predict_probs = model.predict_proba(X_test)
    fpr = [] 
    tpr = []
    for threshold in np.arange(0., 1.01, 0.01):
        tps = (predict_probs[:,1] >= threshold) * (Y_test==1)
        fps = (predict_probs[:,1] >= threshold) *  (Y_test==0)
        
        tpr.append(sum(tps) / tot_pos)
        fpr.append(sum(fps) / tot_neg)
        
    tpr.reverse()
    fpr.reverse()
    plot_roc_curve(fpr, tpr)
    plot_confusion_matrix(prediction, Y_test)
    
    return prediction, fpr, tpr

# In this notebook, we shall use passenger data of the Titanic to investigate the use of classification machine learning to solve problems.

## The data is made up of two parts

- Input *features* (e.g. passenger class, sex, age)
- Output *target*, in this case a binary classification (e.g. *(1,0)* or ('survived', 'died'))

## The aim of a classification problem is one which can be formulated thusly:
### Given data with some given *features*, can we accurately *predict* its *target* as a *category*

In our case, this means we want to be able to predict whether a passenger survived or died given some data description of that passenger.

# Some questions to consider:
## How does this compare to non-machine-learning driven approaches?
## How do we ensure our models and their predictions are robust?
## What are some things we can do to boost our confidence in our machine learning models?

# Let's get started by loading and cleaning the Titanic data.

- "pclass" : Recorded class of the passenger (1st, 2nd, 3rd)
- "sex" : Recorded as Female or Male but we shall change this to (1, 0) respectively
- "age" : Age of adult passengers in years and of children in fraction of years
- "sibsp" : Number of siblings and/or spouses also aboard the Titanic
- "parch" : Number of parents and/or children also aboard the Titanic
- "fare" : The price paid for the passengers ticket at the time in GBP

In [ ]:
# Load the Titanic data in a usable form from Open ML
titanic = fetch_openml(data_id=40945)

data = titanic.data[["pclass","sex", "age", "sibsp", "parch", "fare"]]
target = titanic.target.astype("int32")

In [ ]:
# Renaming "sex" categories to integers to make the data easier to handle by scikitlearn's machinary 
data["sex"] = data["sex"].cat.rename_categories([1, 0])
# Remove data rows with NaN errors
data, target = removeNaNs(data, target)

### If you want to take a look at the data you can by asking Python to print the dataframe

In [ ]:
print(data)

### Or perhaps more usefully for trying to make an initial assessment, we can also plot some of these

To do so, replace "pclass" in the code below with one of the other feature names given in the data table above, e.g "sex" or "fare"


In [ ]:
x_variable = "pclass"

plt.figure()
plt.hist([data[x_variable][target==0], data[x_variable][target==1]], 
         label=["died", "survived"], histtype="barstacked")
plt.xlabel(x_variable)
plt.legend()
plt.show()

# Have a play!
## Can you identify any patterns or even identify selections which you can apply manually to separate the two categories of passengers?
### How "good" can you get?

To do:
- Manual "cut and count" exercise
- Assessment using prediction and plot_confusion_matrix()
- Accuracy score

# You should be able to do a better-than-random job at separating the two categories of passenger on the Titanic, but you should be able to do better with a machine learning approach

Let's start by getting the data split between a *train* and a *test* sample. We do this to validate our model training and ensure that it isn't "overfitting". 

A robustly trained model for predcition (such as a manual scheme for categorising data) should have similar performance on unseen data during testing as with data used in training. Otherwise, we can't be confident in the application of said model. 

In the following, let's split the data features *X* and targets *Y* into 75% for training and 25% for testing of various models.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(data, target, random_state=42)

# Choice of models

All of the models introduced here are *supervised* approaches which can be used in both **classification** and **regression** problems but are discussed in the context of classification only. These models are chosen for their pedagogical simplicity and wealth of documentation available online.

## Support Vector Machines

These models operate by finding the hyperplane(s) best optimised for separating between categories of data. 
The typical pedagogical example is the **linear** support vector machine (SVM) which in 2 dimensions (or for data with only two features which can be plotted on a 2D scatter graph) optimises some line(s) as the decision boundary which best separates the target categories. An example is shown below:

![An example in 2D of a linear Support Vector Machine (SVM) defining a hyperplane for separating between two categories of data with some feature variables (x1,x2)](LinearSVM.webp)
*An example in 2D of a linear Support Vector Machine (SVM) defining a hyperplane for separating between two categories of data with some feature variables (x1,x2). Image taken from an article [here](https://medium.com/@pranavpatil07/support-vector-machine-beginners-guide-7cb678345487).*

\
For an abitrary number of features, N, a linear SVM identifies the N-1 dimensional hyperplane that best separates the target categories based on the input features presented.
While this can be effective enough for some cases, categories in many situations are rarely linearly separable; i.e. in 2D, a straight line may be insufficient. In such cases, a more general class of SVM may be suitable, in which a **kernel**\* is introduced which allows for **non-linear decision boundaries** to be defined. In the 2D case below, the best separation between the two classes is by optimising the circumference of a circle. 

![An example in 2D of a SVM using a kernel to define a non-linear separation between two categories of data with some feature variables (x1,x2)](KernelSVM.webp)
*An example in 2D of a SVM using a kernel to define a non-linear separation between two categories of data with some feature variables (x1,x2). Image taken from an article [here](https://medium.com/@pranavpatil07/support-vector-machine-beginners-guide-7cb678345487).*

\
\* You can think of a kernel as a transformation which allows the SVM to define a more complex separation. For a formal discussion on this, see the resources [here](https://scikit-learn.org/stable/modules/svm.html#svm-kernels).

In scikit-learn SVMs can be used in a versatile way, with a broad choice of kernels (roughly corresponding to different degrees of complexity) available to use. More broadly, SVMs are effective in high-dimensional spaces (large numbers of features) and memory efficient. However, SVMs can be susceptable to over-fitting if kernels are poorly chosen. 

In the cells below, a linear support vector classifier (SVC) is provided as an example alongside a more general classifier which can be setup with other, non-linear kernels.
**You are encouraged to play around with these examples and adjust their parameters.** 
Documentation can be find on [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
or by calling `help()` on a scikit-learn function or object.

In [ ]:
## Initialising the models
LinSVC = LinearSVC() ## Play around with optional hyperparameters here
GenSVC = SVC()

LinSVC.fit(X_train, Y_train)
GenSVC.fit(X_train, Y_train)


print("Linear SVC train score is ", LinSVC.score(X_train, Y_train))
print("Linear SVC test score is ", LinSVC.score(X_test, Y_test))
print("General SVC train score is ", GenSVC.score(X_train, Y_train))
print("General SVC test score is ", GenSVC.score(X_test, Y_test))

## Random forests

Random Forest is an **ensemble** model that fits a number of decision tress and averages results over all the trees. While an individual decision tree is conceptually quite simple, it is also prone to over- or under-fitting. A visualisation for how a decision tree works is shown below:

![Decision Tree Model](DecisionTreeModel.png)
*An example of a 3-deep decision tree using various weather features to determine if one should play or not. Image by T-kita at English Wikipedia*

\
By averaging over many decision trees, a random forest leverages the **power** of decision trees while supporting them to behave more robustly and consistently (hopefully). Reading a single decision tree is an example of an easy interpretation (can be read like a flow chart) while the random forest is harder to interpret because of the aggregation of many trees.

When defining a random forest model, it is important to consider the number of trees (`n_estimators`), how trees are allowed to evolve (`max_depth`, `min_samples_split`, `min_samples_leaf`), and the criterion for optimising target classifier separation (`criterion`).
Typically, individual trees are kept shallow (small number of layers) while the number of trees are made large such that the model is trained to behave generally. However, variation on this may be adjusted in order to learn greater complexity or detail from the data.

In [ ]:
RFclassifier = RandomForestClassifier()

RFclassifier.fit(X_train, Y_train)

print("RF classifier train score is ", RFclassifier.score(X_train, Y_train))
print("RF classifier test score is ", RFclassifier.score(X_test, Y_test))

## Bagging and boosting

The Random Forest described above is an example of **bagging**. This means that generalisation is targeted by training many independent models each on some *subsample* of data, thus reducing the variance in the final aggregate model.

Another approach to reduce overfitting  and train a generalisable model is **boosting**. Rather than train models independently, models are trained one-by-one, each time correcting for misclassification in the previous one. This reduces the bias in the model and trains the model to be more influenced by better performing models. 

In practice, modern machine learning approaches to problems typically use some combination of both bagging and boosting to design robust machine learning models.

In scikit-learn, it is possible to introduce boosting in various ways. Below, AdaBoost (or adaptive boosting) is used on a DecisionTreeClassifier. The `n_estimators` parameter used in the `adaboost_clf` definition can also be adjusted to see how the boosting behaves on a single decision tree (`n=1`) compared to a forest of trees (e.g. `n=100`).

In [ ]:
## Define (some number of) a weak learner to be used in the boosted model
weak_learner = DecisionTreeClassifier(max_leaf_nodes=8)
n_estimators = 300

## Define the Adaptive Boosting algorithm
adaboost_clf = AdaBoostClassifier(
    estimator=weak_learner,
    n_estimators=n_estimators,
    random_state=42,
)

adaboost_clf.fit(X_train, Y_train)

print("Gradiant boosted decision tree train score is ", adaboost_clf.score(X_train, Y_train))
print("Gradiant boosted decision tree test score is ", adaboost_clf.score(X_test, Y_test))

# Have a play!

- What do you notice, which model(s) perform better, which perform worse?
- How do results for the ML models compare to results acheived through a more manual approach (e.g some decided selection cuts based on what is seen in plots of the data)?
- What is the highest accuracy which you can get?
- How do results using training and testing scores differ?

# Assessing and optimising a model

## I've trained my model, but how do I know it is actually doing a good job?

While we have thus far considered the raw accuracy as a metric, it is important that this used properly and in conjuction with other robust indicators of model performance. 

Naturally, a more accurate score is desirable as it indicates that a model is more powerful at classifying data into its categories. However, it is perhaps more important to know that a model is consistent in its behaviour - i.e are we confident that the model is robust at predicting the target class when used on unseen data as in training. 

Therefore, if the accuracy of the model varies wildly between training and testing, this indicates an undesirable model performance, usually the result of overfitting on the training data.

### So while getting started with classification models on scikit-learn can be relatively simple, producing results which are robust and understood requires a bit more care

## ROC curves and confusion matrices

To understand the trained models in more depth, it is a good idea to understand their true- and false-positive rates. If this is familiar, that is because this is related to understanding type 1 (false positive, $\alpha$) and type 2 (false negative, $\beta$) errors. Depending on the situation upon which machine learning models are being applied, maximising true-positive rates may or may not be more important than minimising false-postive rates.

One way in which this can be considered is by plotting a confusion matrix which shows the rates prediction and truth match each other or not. In a perfect model, there are no type 1 nor type 2 errors, therefore the true-positive rate is 100%. This means that the matrix shown below would be perfectly diagonal.

Another way of considering model performance is by plotting the true- and false-positive rates (tpr and fpr respectively) against each other in something called a receiver operator characteristic or **ROC curve**.

In a perfect classifier the predicted output is perfectly binary (e.g. `[0,1]` corresponding to two classes). In reality, data is *messy* so the categories overlap. This means that real classifiers output probabilities that given data belongs to some category (e.g. `[0.39, 0.61]`).
Therefore, the assigned classification for given data belonging to one category over another (e.g. 'dead' or 'survived') is based on some **threshold**.

By default, this simply assigns data to the category with maximum probability. However, it is also possible to vary this. By scanning over this threshold, we can examine how the fpr evolves as a function of the tpr.

An example of these two metrics in action is shown below for the random forest classifier:

In [ ]:
prediction, fpr, tpr = roc_curve(RFclassifier, X_train, Y_train)

In [ ]:
prediction, fpr, tpr = roc_curve(RFclassifier, X_test, Y_test)

In the conventions given above, perfect classifiers should have:
- confusion matrices with all their entries on the positive diagonal
- a square ROC with its upper left corner at `(0., 1.)`.

In considering whether the model is robust, the question thus reduces to the following:
- Are ROC curves similar between testing and training?
- Is my type 1 and type 2 error seen in the confusion matrix consistent between training and testing?

# So with this in mind, can you use the power of classification machine learning to predict whether a passenger survives the Titanic disaster?

## Good luck

You may wish to try the following exercises:
1. Examine the ROC curves and confusion matrices for all the model architectures considered in this notebook
2. Test different hyperparameters (settings in the scikit-learn object construction calls) SVCs and decision tree models. 
3. Have a look into other models and features on the [scikit-learn documentation](https://scikit-learn.org/stable/supervised_learning.html).
4. Test what you have learned on a problem of your own or go onto [Kaggle](https://www.kaggle.com) to have a play!